# Coleta Completa de Noticias Multi-Source — ITUB4

Pipeline completo para Google Colab:
1. Instala dependencias
2. Coleta noticias de 3 fontes (InfoMoney, CVM, Google News)
3. Roda FinBERT-PT-BR para analise de sentimento
4. Gera CSV de sentimento diario (compativel com o pipeline de treino)

**Fontes:**
- InfoMoney (WordPress API) — artigos completos, historico extenso
- CVM Fatos Relevantes (dados abertos) — comunicados oficiais desde 2009
- Google News (RSS + gnews) — agregador de multiplas fontes (~1 ano)

In [ ]:
!pip install gnews feedparser googlenewsdecoder newspaper3k lxml_html_clean transformers torch yfinance -q
print("Dependencias instaladas!")

## Configuracao

Defina o ticker e o periodo de coleta.

In [ ]:
# === CONFIGURACAO ===
TICKER = "ITUB4"
TICKER_YAHOO = "ITUB4.SA"
COMPANY_NAME = "Itaú Unibanco"

# InfoMoney: periodo em meses para tras (maximo disponivel: ~200 meses / ~17 anos)
INFOMONEY_MESES = 200

# CVM: anos para coleta de Fatos Relevantes
CVM_ANO_INICIO = 2009
CVM_ANO_FIM = 2026

# Google News: periodo (limitado pela API)
GNEWS_PERIOD = "1y"  # opcoes: "1h", "1d", "7d", "1m", "6m", "1y"
GNEWS_MAX = 200

# FinBERT: modelo
FINBERT_MODEL = "lucas-leme/FinBERT-PT-BR"

print(f"Ticker: {TICKER}")
print(f"InfoMoney: ultimos {INFOMONEY_MESES} meses")
print(f"CVM: {CVM_ANO_INICIO} a {CVM_ANO_FIM}")
print(f"Google News: ultimo {GNEWS_PERIOD}")

## 1. Coleta — InfoMoney

Usa a REST API do WordPress do InfoMoney para buscar todos os artigos sobre o ticker.
A API permite buscar ate 100 artigos por pagina, com paginacao automatica.

In [ ]:
import requests
import json
import re
import time
from html import unescape
from datetime import datetime, timedelta

def strip_html(html_text):
    """Remove tags HTML e decodifica entidades."""
    text = re.sub(r"<[^>]+>", "", html_text)
    text = unescape(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def collect_infomoney(ticker, meses_atras=12, per_page=100):
    """
    Coleta artigos do InfoMoney via WordPress REST API.
    
    Usa paginacao automatica e retentativas com backoff.
    Retorna lista de dicts com: date, title, text, source, link
    """
    base_url = "https://www.infomoney.com.br/wp-json/wp/v2/posts"
    headers = {
        "accept": "application/json",
        "Referer": "https://www.infomoney.com.br/",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    }
    
    data_fim = datetime.now()
    data_inicio = data_fim - timedelta(days=30 * meses_atras)
    modified_after = data_inicio.strftime("%Y-%m-%dT%H:%M:%S")
    
    print(f"[InfoMoney] Buscando artigos sobre {ticker}")
    print(f"  Periodo: {data_inicio.strftime('%Y-%m-%d')} a {data_fim.strftime('%Y-%m-%d')}")
    
    articles = []
    ids_vistos = set()
    page = 1
    
    while True:
        params = {
            "per_page": min(per_page, 100),
            "page": page,
            "search": ticker,
            "orderby": "modified",
            "order": "desc",
            "status": "publish",
            "modified_after": modified_after,
        }
        
        for attempt in range(3):
            try:
                resp = requests.get(base_url, headers=headers, params=params, timeout=20)
                if resp.status_code == 400:
                    break
                resp.raise_for_status()
                break
            except Exception as e:
                if attempt == 2:
                    print(f"  ERRO pagina {page}: {e}")
                    break
                time.sleep(attempt + 1)
        
        if resp.status_code == 400:
            break
        
        data = resp.json()
        if not data:
            break
        
        for raw in data:
            art_id = raw.get("id", 0)
            if art_id in ids_vistos:
                continue
            ids_vistos.add(art_id)
            
            title = strip_html(raw.get("title", {}).get("rendered", ""))
            excerpt = strip_html(raw.get("excerpt", {}).get("rendered", ""))
            content = strip_html(raw.get("content", {}).get("rendered", ""))
            
            articles.append({
                "date": raw.get("date", "")[:10],
                "title": title,
                "text": f"{title}. {excerpt}. {content}"[:50000],
                "source": "InfoMoney",
                "link": raw.get("link", ""),
            })
        
        total_pages = int(resp.headers.get("X-WP-TotalPages", 1))
        print(f"  Pagina {page}/{total_pages} — {len(articles)} artigos acumulados")
        
        if page >= total_pages:
            break
        page += 1
        time.sleep(0.5)  # delay entre paginas
    
    print(f"[InfoMoney] Total: {len(articles)} artigos coletados")
    return articles

# === EXECUTAR ===
infomoney_articles = collect_infomoney(TICKER, meses_atras=INFOMONEY_MESES)

## 2. Coleta — CVM Fatos Relevantes

Baixa os dados abertos da CVM (Comissao de Valores Mobiliarios).
Cada ano vem como um ZIP com CSVs. Filtramos por "ITAU UNIBANCO" + "Fato Relevante".

In [ ]:
import pandas as pd
import zipfile
import io
from bs4 import BeautifulSoup

def collect_cvm(start_year=2009, end_year=2026, company="ITAU"):
    """
    Coleta Fatos Relevantes da CVM para uma empresa.
    
    Baixa ZIPs anuais de https://dados.cvm.gov.br, filtra por empresa
    e tipo de documento, e extrai o texto de cada fato relevante.
    """
    base_url = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/IPE/DADOS/ipe_cia_aberta_{year}.zip"
    articles = []
    
    for year in range(start_year, end_year + 1):
        url = base_url.format(year=year)
        print(f"[CVM] Baixando {year}...", end=" ")
        
        try:
            resp = requests.get(url, timeout=60)
            resp.raise_for_status()
        except Exception as e:
            print(f"ERRO: {e}")
            continue
        
        # Descompactar ZIP e ler CSVs
        frames = []
        with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
            for name in zf.namelist():
                if name.endswith(".csv"):
                    with zf.open(name) as f:
                        df = pd.read_csv(f, sep=";", encoding="latin-1", dtype=str, on_bad_lines="skip")
                        frames.append(df)
        
        if not frames:
            print("vazio")
            continue
        
        data = pd.concat(frames, ignore_index=True)
        
        # Filtrar por empresa e tipo de documento
        # Colunas podem variar: Nome_Companhia ou DENOM_CIA
        cia_col = "Nome_Companhia" if "Nome_Companhia" in data.columns else "DENOM_CIA"
        cat_col = "Categoria" if "Categoria" in data.columns else "CATEG_DOC"
        date_col = "Data_Referencia" if "Data_Referencia" in data.columns else "DT_REFER"
        link_col = "Link_Download" if "Link_Download" in data.columns else "LINK_DOC"
        
        mask_cia = data[cia_col].str.contains(company, case=False, na=False)
        mask_cia = mask_cia & data[cia_col].str.contains("UNIBANCO", case=False, na=False)
        mask_doc = data[cat_col].str.contains("Fato Relevante", case=False, na=False)
        
        filtered = data[mask_cia & mask_doc]
        print(f"{len(filtered)} fatos relevantes")
        
        # Extrair texto de cada documento
        for _, row in filtered.iterrows():
            date_str = row.get(date_col, "")
            try:
                date_iso = pd.to_datetime(date_str).strftime("%Y-%m-%d")
            except:
                date_iso = str(date_str)
            
            link = row.get(link_col, "")
            text = ""
            if link:
                try:
                    r = requests.get(link, timeout=30)
                    r.encoding = r.apparent_encoding or "latin-1"
                    soup = BeautifulSoup(r.text, "html.parser")
                    for tag in soup(["script", "style"]):
                        tag.decompose()
                    text = soup.get_text(separator="
", strip=True)
                except:
                    pass
                time.sleep(0.5)
            
            articles.append({
                "date": date_iso,
                "title": "Fato Relevante",
                "text": text[:50000] if text else "Fato Relevante",
                "source": "CVM",
                "link": link,
            })
    
    print(f"[CVM] Total: {len(articles)} fatos relevantes coletados")
    return articles

# === EXECUTAR ===
cvm_articles = collect_cvm(start_year=CVM_ANO_INICIO, end_year=CVM_ANO_FIM)

## 3. Coleta — Google News

Busca noticias de multiplas fontes via Google News (Valor Economico, Reuters, Bloomberg, Exame, etc.).
Tenta extrair o texto completo de cada artigo decodificando a URL do Google News.

In [ ]:
from gnews import GNews
import feedparser
from urllib.parse import quote
from googlenewsdecoder import new_decoderv1
from newspaper import Article

def collect_google_news(queries, period="1y", max_results=200):
    """
    Coleta noticias do Google News via gnews + RSS.
    Decodifica URLs do Google News e extrai texto completo quando possivel.
    """
    all_raw = []
    
    for query in queries:
        # gnews
        try:
            gn = GNews(language="pt", country="BR", period=period, max_results=max_results)
            arts = gn.get_news(query)
            print(f"[Google News] gnews '{query}': {len(arts)} artigos")
            for a in arts:
                pub = a.get("published date", "")
                try:
                    dt = datetime.strptime(pub, "%a, %d %b %Y %H:%M:%S %Z")
                    date_str = dt.strftime("%Y-%m-%d")
                except:
                    date_str = pub
                
                publisher = a.get("publisher", {})
                pub_name = publisher.get("title", "Desconhecido") if isinstance(publisher, dict) else str(publisher)
                
                all_raw.append({
                    "date": date_str,
                    "title": a.get("title", ""),
                    "link": a.get("url", ""),
                    "source": pub_name,
                })
        except Exception as e:
            print(f"[Google News] gnews '{query}' ERRO: {e}")
        
        # RSS
        try:
            url = f"https://news.google.com/rss/search?q={quote(query)}&hl=pt-BR&gl=BR&ceid=BR:pt-419"
            feed = feedparser.parse(url)
            print(f"[Google News] RSS '{query}': {len(feed.entries)} artigos")
            for entry in feed.entries:
                pub = entry.get("published", "")
                try:
                    dt = datetime.strptime(pub, "%a, %d %b %Y %H:%M:%S %Z")
                    date_str = dt.strftime("%Y-%m-%d")
                except:
                    date_str = pub
                
                source = entry.get("source", {})
                src_name = source.get("title", "Desconhecido") if isinstance(source, dict) else str(source) if source else "Desconhecido"
                
                all_raw.append({
                    "date": date_str,
                    "title": re.sub(r"<[^>]+>", "", entry.get("title", "")),
                    "link": entry.get("link", ""),
                    "source": src_name,
                })
        except Exception as e:
            print(f"[Google News] RSS '{query}' ERRO: {e}")
    
    # Deduplicar por URL
    seen = set()
    unique = []
    for a in all_raw:
        if a["link"] not in seen:
            seen.add(a["link"])
            unique.append(a)
    
    print(f"[Google News] {len(unique)} artigos unicos apos deduplicacao")
    
    # Extrair texto completo
    articles = []
    success = 0
    for i, a in enumerate(unique):
        title = re.sub(r"<[^>]+>", "", a["title"]).strip()
        
        # Decodificar URL real do Google News
        full_text = None
        real_url = a["link"]
        try:
            result = new_decoderv1(a["link"])
            if result.get("status"):
                real_url = result["decoded_url"]
                # Extrair texto do site original
                art = Article(real_url, language="pt")
                art.download()
                art.parse()
                if art.text and len(art.text) > 50:
                    full_text = art.text
                    success += 1
        except:
            pass
        
        articles.append({
            "date": a["date"],
            "title": title,
            "text": full_text[:50000] if full_text else title,
            "source": a["source"],
            "link": real_url,
            "has_full_text": full_text is not None,
        })
        
        if (i + 1) % 20 == 0:
            print(f"  Progresso: {i+1}/{len(unique)} (texto completo: {success})")
        time.sleep(1.0)
    
    print(f"[Google News] Total: {len(articles)} artigos ({success} com texto completo)")
    return articles

# === EXECUTAR ===
queries = [TICKER, COMPANY_NAME, f"{COMPANY_NAME} acoes"]
gnews_articles = collect_google_news(queries, period=GNEWS_PERIOD, max_results=GNEWS_MAX)

## 4. Combinar todas as fontes

In [ ]:
import numpy as np

all_articles = infomoney_articles + cvm_articles + gnews_articles

print(f"{'='*60}")
print(f"TOTAL: {len(all_articles)} artigos de todas as fontes")
print(f"{'='*60}")
print(f"  InfoMoney:    {len(infomoney_articles)}")
print(f"  CVM:          {len(cvm_articles)}")
print(f"  Google News:  {len(gnews_articles)}")

# Resumo por data
df_all = pd.DataFrame(all_articles)
df_all['date'] = pd.to_datetime(df_all['date'], errors='coerce')
df_all = df_all.dropna(subset=['date'])
print(f"
Periodo: {df_all['date'].min().date()} a {df_all['date'].max().date()}")
print(f"
Top 10 fontes:")
print(df_all['source'].value_counts().head(10).to_string())

## 5. Analise de sentimento — FinBERT-PT-BR

Carrega o modelo FinBERT treinado em textos financeiros brasileiros e analisa
o sentimento de cada artigo. Gera 3 logits (positivo, negativo, neutro) e
uma classe final por artigo.

In [ ]:
import torch
from transformers import AutoTokenizer, BertForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL)
model = BertForSequenceClassification.from_pretrained(FINBERT_MODEL)
model.eval()

LABEL_MAP = {0: 'POSITIVE', 1: 'NEGATIVE', 2: 'NEUTRAL'}
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f'FinBERT carregado no device: {device}')

In [ ]:
def predict_sentiment(texts, batch_size=32):
    """Analisa sentimento de textos com FinBERT em batches."""
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        batch = [t[:512] if t else "" for t in batch]
        
        tokens = tokenizer(batch, return_tensors='pt', padding=True,
                           truncation=True, max_length=512).to(device)
        with torch.no_grad():
            logits = model(**tokens).logits.cpu().numpy()
        
        for log in logits:
            cls = int(np.argmax(log))
            results.append({
                'sentiment_class': cls,
                'sentiment': LABEL_MAP[cls],
                'logits': log.tolist(),
            })
        
        if (i + batch_size) % 100 == 0:
            print(f"  Progresso: {min(i+batch_size, len(texts))}/{len(texts)}")
    
    return results

# Preparar textos
texts = [a.get('text', '') or a.get('title', '') for a in all_articles]
print(f"Analisando sentimento de {len(texts)} artigos...")

sentiments = predict_sentiment(texts)

# Adicionar sentimento aos artigos
for art, sent in zip(all_articles, sentiments):
    art['sentiment_class'] = sent['sentiment_class']
    art['sentiment'] = sent['sentiment']
    art['sentiment_logits'] = sent['logits']

# Resumo
from collections import Counter
counts = Counter(s['sentiment'] for s in sentiments)
print(f"
Distribuicao de sentimento:")
for label, count in counts.most_common():
    print(f"  {label}: {count} ({count/len(sentiments)*100:.1f}%)")

## 6. Agregacao diaria de sentimento

Gera o CSV de sentimento diario com as mesmas colunas usadas nos Stages 4-6:
- `n_articles`: quantidade de artigos no dia
- `mean_logit_pos`: media do logit positivo
- `mean_logit_neg`: media do logit negativo
- `mean_logit_neu`: media do logit neutro
- `mean_sentiment`: media da classe de sentimento

In [ ]:
df_sent = pd.DataFrame([{
    'date': a['date'],
    'source': a.get('source', ''),
    'sentiment_class': a['sentiment_class'],
    'logit_pos': a['sentiment_logits'][0],
    'logit_neg': a['sentiment_logits'][1],
    'logit_neu': a['sentiment_logits'][2],
} for a in all_articles])

df_sent['date'] = pd.to_datetime(df_sent['date'], errors='coerce')
df_sent = df_sent.dropna(subset=['date'])

# Agregacao diaria
daily = df_sent.groupby('date').agg(
    n_articles=('sentiment_class', 'count'),
    mean_logit_pos=('logit_pos', 'mean'),
    mean_logit_neg=('logit_neg', 'mean'),
    mean_logit_neu=('logit_neu', 'mean'),
    mean_sentiment=('sentiment_class', 'mean'),
).sort_index()

print(f"Sentimento diario: {len(daily)} dias")
print(f"Periodo: {daily.index[0].date()} a {daily.index[-1].date()}")
print(f"
Ultimos 10 dias:")
display(daily.tail(10))

## 7. Salvar resultados

In [ ]:
# Salvar sentimento diario (CSV compativel com pipeline de treino)
daily.to_csv('multi_source_daily_sentiment.csv')
print(f"Sentimento diario salvo: multi_source_daily_sentiment.csv ({len(daily)} dias)")

# Salvar artigos completos com sentimento (JSON para referencia)
with open('all_articles_with_sentiment.json', 'w', encoding='utf-8') as f:
    for a in all_articles:
        if hasattr(a.get('date'), 'strftime'):
            a['date'] = a['date'].strftime('%Y-%m-%d')
    json.dump(all_articles, f, ensure_ascii=False, indent=2, default=str)
print(f"Artigos salvos: all_articles_with_sentiment.json ({len(all_articles)} artigos)")

# Resumo final
print(f"
{'='*60}")
print(f"COLETA COMPLETA")
print(f"{'='*60}")
print(f"Artigos totais: {len(all_articles)}")
print(f"  InfoMoney: {len(infomoney_articles)}")
print(f"  CVM: {len(cvm_articles)}")
print(f"  Google News: {len(gnews_articles)}")
print(f"Dias com sentimento: {len(daily)}")
print(f"Periodo: {daily.index[0].date()} a {daily.index[-1].date()}")

## 8. (Opcional) Salvar no Google Drive

Descomente as linhas abaixo para salvar os resultados no seu Google Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# 
# import shutil
# dest = '/content/drive/MyDrive/tcc-takeo/8.multi-source-news/'
# !mkdir -p "{dest}"
# shutil.copy('multi_source_daily_sentiment.csv', dest)
# shutil.copy('all_articles_with_sentiment.json', dest)
# print(f"Arquivos salvos em {dest}")